In [ ]:
import argparse
import os
import sys
import cv2
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
import torch.nn as nn
from tqdm import tqdm
from utils.dataloader_cls import BUS_CEUS_Classification, BUS_CEUS_Classification_Images
from models.tsnet import tsnet_rx18, tsnet_rx50
from models.asycmst import AsyCMST
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score, roc_auc_score
from utils.eval_metrics import confusion_matrix, draw_roc_curve
import datetime
from torch.utils.tensorboard import SummaryWriter
import yaml
from easydict import EasyDict as edict

In [ ]:
# config_path = './config/train_tsnet_rx50_bus_ceus.yaml'
config_path = './config/train_asycmst_bus_ceus.yaml'
config = edict(yaml.load(open(config_path), Loader=yaml.FullLoader))

model_dict = {
              'tsnet_rx18': tsnet_rx18(num_classes=2, sample_size=224, sample_duration=16),
              'tsnet_rx50': tsnet_rx50(num_classes=2, sample_size=224, sample_duration=16),
              'asycmst_r18_384_6_6': AsyCMST(num_classes=2, sample_size=224, sample_duration=16, embed_dim=384, num_layers=6, num_heads=6)
              }

task = config.task
model_name = config.model.name
device = torch.device(config.device.single if torch.cuda.is_available() else "cpu")

data_root_dir = config.data.root_dir
train_label = config.data.train_label
valid_label = config.data.valid_label
test_label = config.data.test_label

n_classes = config.data.num_class
batch_size = config.params.batch_size
num_epoches = config.params.epoch
learning_rate = config.params.lr

vid_size = config.data.size
num_frm = config.data.num_frm

net = model_dict[model_name].to(device)
optimizer = torch.optim.RMSprop(net.parameters(), lr=learning_rate, momentum=0.9, weight_decay=1e-4)
# optimizer = torch.optim.Adam(net.parameters(), lr=learning_rate, weight_decay=1e-4, betas=(0.9, 0.999), eps=1e-08)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', factor=0.5, patience=3)
loss_weight = torch.tensor([1.0, 1.0])
criterion = nn.CrossEntropyLoss(loss_weight).to(device)
timestamp = datetime.datetime.now().strftime('%Y_%m_%d_%H_%M_%S')
save_name = f'{timestamp}_{model_name}'
writer_dir = os.path.join('runs', task, save_name)
writer = SummaryWriter(log_dir=writer_dir)
save_dir = os.path.join(writer_dir, 'ckpt')
log_dir = os.path.join(writer_dir, 'logs')

try:
    os.makedirs(save_dir)
except:
    pass
try:
    os.makedirs(log_dir)
except:
    pass

train_dataset = BUS_CEUS_Classification_Images(
    root_dir=data_root_dir, 
    label_csv=train_label,
    video_size=vid_size,
    num_frm=num_frm,
    mode='train',
    augment=config.data.augment)

valid_dataset = BUS_CEUS_Classification_Images(
    root_dir=data_root_dir, 
    label_csv=valid_label,
    video_size=vid_size,
    num_frm=num_frm,
    mode='eval',
    augment=config.data.augment)

train_num = len(train_dataset)
valid_num = len(valid_dataset)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size, 
    shuffle=True,
    num_workers=8,
    drop_last=True)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=batch_size, 
    shuffle=False,
    num_workers=8)

print(f"Using {train_num} samples for training, {valid_num} samples for validation, {n_classes} classes.")

In [ ]:
global_step = 0
best_acc = [0.0]*3

for epoch in range(num_epoches):
    
    # train
    net.train()
    epoch_loss = 0.0
    pred_list = []
    pred_score_list = []
    label_list = []
    with tqdm(total=len(train_dataset), desc=f'Epoch {epoch+1}', unit='imgs') as pbar:
        for batch in train_loader:
            images_ceus, images_bus, labels = batch['segment_ceus'], batch['segment_bus'], batch['label']
            images_ceus, images_bus = images_ceus.to(device), images_bus.to(device)
            labels = labels.to(device)
            logits = net(images_bus, images_ceus)
            loss = criterion(logits, labels)
            epoch_loss += loss.detach().item()
            pred_score = torch.softmax(logits, dim=1)
            pred_results = torch.argmax(pred_score, dim=1)
            pred_list.extend(pred_results.tolist())
            pred_score_list.extend(pred_score[:,1].tolist())
            label_list.extend(labels.tolist())
            
            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_value_(net.parameters(), 0.1)
            optimizer.step()
            pbar.update(images_bus.shape[0])
            global_step += 1

    train_loss = epoch_loss / (len(train_dataset)//batch_size*batch_size)
    train_acc = accuracy_score(label_list, pred_list)
    train_rec = recall_score(label_list, pred_list)
    train_pre = precision_score(label_list, pred_list, zero_division=0)
    train_f1 = f1_score(label_list, pred_list)
    train_auc = roc_auc_score(label_list, pred_score_list)
    draw_roc_curve(label_list, pred_score_list, (2.5,2.5), os.path.join(log_dir, f'Train_Epoch_{epoch}_Acc_{train_acc:.4f}_RoC.jpg'), 'Train')
    train_cm = confusion_matrix(label_list, pred_list, [str(i) for i in range(n_classes)], (2.5,2), os.path.join(log_dir, f'Train_Epoch_{epoch}_Acc_{train_acc:.4f}_CM.jpg'))
    train_df = pd.DataFrame({'score': pred_score_list, 'pred': pred_list, 'label': label_list})
    train_df.to_csv(os.path.join(log_dir, f'Train_Epoch_{epoch}_Acc_{train_acc:.4f}.csv'), index=False)
    scheduler.step(train_loss)
    
    writer.add_scalar('0.Loss/train', train_loss, global_step)
    writer.add_scalar('1.Acc/train', train_acc, global_step)
    writer.add_scalar('2.Rec/train', train_rec, global_step)
    writer.add_scalar('3.Pre/train', train_pre, global_step)
    writer.add_scalar('4.F1/train', train_f1, global_step)
    writer.add_scalar('5.AUC/train', train_auc, global_step)
    writer.add_scalar('6.Learning rate', optimizer.param_groups[0]['lr'], global_step)
    
    # validation
    net.eval()
    with tqdm(total=len(valid_dataset), desc=f'Validation', unit='imgs') as pbar_valid:
        valid_loss = 0.0
        pred_list = []
        pred_score_list = []
        label_list = []
        for batch in valid_loader:
            images_ceus, images_bus, labels = batch['segment_ceus'], batch['segment_bus'], batch['label']
            images_ceus, images_bus = images_ceus.to(device), images_bus.to(device)
            labels = labels.to(device)
            with torch.no_grad():
                logits = net(images_bus, images_ceus)
                loss = criterion(logits, labels)
            epoch_loss += loss.detach().item()
            pred_score = torch.softmax(logits, dim=1)
            pred_results = torch.argmax(pred_score, dim=1)
            pred_list.extend(pred_results.tolist())
            pred_score_list.extend(pred_score[:,1].tolist())
            label_list.extend(labels.tolist())
            pbar_valid.update(images_bus.shape[0])
    
    valid_loss = epoch_loss / len(valid_dataset)
    valid_acc = accuracy_score(label_list, pred_list)
    valid_rec = recall_score(label_list, pred_list)
    valid_pre = precision_score(label_list, pred_list, zero_division=0)
    valid_f1 = f1_score(label_list, pred_list)
    valid_auc = roc_auc_score(label_list, pred_score_list)
    draw_roc_curve(label_list, pred_score_list, (2.5,2.5), os.path.join(log_dir, f'Valid_Epoch_{epoch}_Acc_{valid_acc:.4f}_RoC.jpg'), 'Valid')
    valid_cm = confusion_matrix(label_list, pred_list, [str(i) for i in range(n_classes)], (2.5,2), os.path.join(log_dir, f'Valid_Epoch_{epoch}_Acc_{valid_acc:.4f}_CM.jpg'))
    valid_df = pd.DataFrame({'score': pred_score_list, 'pred': pred_list, 'label': label_list})
    valid_df.to_csv(os.path.join(log_dir, f'Valid_Epoch_{epoch}_Acc_{valid_acc:.4f}.csv'), index=False)
    
    writer.add_scalar('0.Loss/valid', valid_loss, global_step)
    writer.add_scalar('1.Acc/valid', valid_acc, global_step)
    writer.add_scalar('2.Rec/valid', valid_rec, global_step)
    writer.add_scalar('3.Pre/valid', valid_pre, global_step)
    writer.add_scalar('4.F1/valid', valid_f1, global_step)
    writer.add_scalar('5.AUC/valid', valid_auc, global_step)
    
    if valid_acc >= best_acc[0]:
        torch.save(net.state_dict(), os.path.join(save_dir, f'val_acc_{valid_acc:.4f}_epoch_{epoch:03d}.pth'))
        print('ckpt saved!')
        best_acc[0] = valid_acc
        best_acc.sort()
        if len(os.listdir(save_dir)) > 3:
            ckpt_list = os.listdir(save_dir)
            ckpt_list.sort()
            os.remove(os.path.join(save_dir, ckpt_list[0]))
    
    print(f'''Train: loss:{train_loss:.8f}, Acc: {train_acc:.4f}, F1: {train_f1:.4f}, AUC: {train_auc:.4f}
Valid: loss:{valid_loss:.8f}, Acc: {valid_acc:.4f}, F1: {valid_f1:.4f}, AUC: {valid_auc:.4f}
LR:{optimizer.param_groups[0]['lr']}''')

writer.close()

In [ ]:
import os
import torch

torch.cuda.empty_cache()
# pid = os.getpid()
# !kill -9 $pid